# Sesi 3 — 3.3 Data Transformation

Isi bagian `# TODO:` sesuai instruksi di tiap cell

Input: `orders_cleaned_intermediate.csv` hasil dari bagian 3.2


## 1. Setup + load


In [10]:
import pandas as pd
from pathlib import Path

# TODO: load orders_cleaned_intermediate.csv (fallback ke orders_clean.csv kalau tidak ada)
path = "../orders_cleaned_intermediate.csv"
if not Path(path).exists():
    path = "../orders_clean.csv"
    print("Fallback ke orders_clean.csv karena orders_cleaned_intermediate.csv tidak ditemukan.")
df = pd.read_csv(path)
print("Shape:", df.shape)
df.head()


Shape: (488, 14)


,Unnamed: 0,order_id,order_date,customer_name,email,segment,city,category,product_name,price_idr,quantity,discount,rating,product_review
0,0,ORD-00059,2024-08-13,Putri Rahman,putri.rahman15@yahoo.com,Consumer,Makassar,rumah,Lampu Meja,814512,3,0.00,3.5,"Recommended seller, packing rapi"
1,1,ORD-00280,2024-03-14,Indah Hidayat,indah.hidayat51@gmail.com,Corporate,Medan,elektronik,Laptop,5686210,10,0.20,4.4,Harga terjangkau dan barang berkualitas
2,2,ORD-00342,2024-10-16,Sari Pratama,sari.pratama32@yahoo.com,Consumer,Makassar,makanan,Teh Celup,105549,6,0.10,3.3,"Puas dengan pembelian ini, akan beli lagi"
3,3,ORD-00430,2024-11-22,Oki Lestari,oki.lestari24@gmail.com,Consumer,Malang,kesehatan,Hand Sanitizer,168548,3,0.25,3.4,Sesuai ekspektasi
4,4,ORD-00367,2024-06-27,Rina Utami,rina.utami59@gmail.com,Corporate,Malang,makanan,Teh Celup,101088,5,0.00,4.2,"Pengiriman cepat, kualitas mantap"


## 2. MinMax Scaling


In [11]:
from sklearn.preprocessing import MinMaxScaler

# TODO: buat scaler, apply ke kolom [price_idr, rating]
print("Sebelum scaling:")
print(df[['price_idr', 'rating']].agg(['min', 'max']))

scaler = MinMaxScaler()
df[['price_norm', 'rating_norm']] = scaler.fit_transform(df[['price_idr', 'rating']])
print("Setelah scaling:")
print(df[['price_norm', 'rating_norm']].agg(['min', 'max']))

Sebelum scaling:
     price_idr  rating
min      12997     3.0
max    7765195     5.0
Setelah scaling:
     price_norm  rating_norm
min         0.0          0.0
max         1.0          1.0


In [12]:
df['category'].value_counts()

category
rumah         109
makanan       107
kesehatan      96
elektronik     93
fashion        83
Name: count, dtype: int64

In [13]:
# TODO: cek range setelah scaling — semua harus 0–1
print("Range setelah scaling:")
print(df[['price_norm', 'rating_norm']].agg(['min', 'max']))

Range setelah scaling:
     price_norm  rating_norm
min         0.0          0.0
max         1.0          1.0


## 3. One-Hot Encoding kategori


In [14]:
# TODO: pakai pd.get_dummies pada kolom "category" dengan prefix "cat"
df_encoded = pd.get_dummies(df, columns=['category'], prefix='cat')
# Lihat kolom baru yang muncul 
df_encoded.head()

,Unnamed: 0,order_id,order_date,customer_name,email,segment,city,product_name,price_idr,quantity,discount,rating,product_review,price_norm,rating_norm,cat_elektronik,cat_fashion,cat_kesehatan,cat_makanan,cat_rumah
0,0,ORD-00059,2024-08-13,Putri Rahman,putri.rahman15@yahoo.com,Consumer,Makassar,Lampu Meja,814512,3,0.00,3.5,"Recommended seller, packing rapi",0.103392,0.25,False,False,False,False,True
1,1,ORD-00280,2024-03-14,Indah Hidayat,indah.hidayat51@gmail.com,Corporate,Medan,Laptop,5686210,10,0.20,4.4,Harga terjangkau dan barang berkualitas,0.731820,0.70,True,False,False,False,False
2,2,ORD-00342,2024-10-16,Sari Pratama,sari.pratama32@yahoo.com,Consumer,Makassar,Teh Celup,105549,6,0.10,3.3,"Puas dengan pembelian ini, akan beli lagi",0.011939,0.15,False,False,False,True,False
3,3,ORD-00430,2024-11-22,Oki Lestari,oki.lestari24@gmail.com,Consumer,Malang,Hand Sanitizer,168548,3,0.25,3.4,Sesuai ekspektasi,0.020065,0.20,False,False,True,False,False
4,4,ORD-00367,2024-06-27,Rina Utami,rina.utami59@gmail.com,Corporate,Malang,Teh Celup,101088,5,0.00,4.2,"Pengiriman cepat, kualitas mantap",0.011363,0.60,False,False,False,True,False


## 4. Text cleaning review


In [15]:
# TODO: buat kolom review_clean:
# - str.lower()
# - str.replace tanda baca dengan regex r"[^\w\s]"
# - str.strip()

df_encoded['review_clean'] = (
    df_encoded['product_review']
    .astype(str) # tipe data pastika string
    .str.lower() # ubah ke huruf menjadi kecil
    .str.replace(r"[^\w\s]", "", regex=True) # hapus tanda baca seperti !?,.
    .str.strip() # menghapus whitespace di awal dan akhir string
)

In [17]:
df_encoded[['product_review', 'review_clean']].head(10)

,product_review,review_clean
0,"Recommended seller, packing rapi",recommended seller packing rapi
1,Harga terjangkau dan barang berkualitas,harga terjangkau dan barang berkualitas
2,"Puas dengan pembelian ini, akan beli lagi",puas dengan pembelian ini akan beli lagi
3,Sesuai ekspektasi,sesuai ekspektasi
4,"Pengiriman cepat, kualitas mantap",pengiriman cepat kualitas mantap
5,Barang sampai dengan baik,barang sampai dengan baik
6,Ada sedikit cacat pada produk,ada sedikit cacat pada produk
7,"Pengiriman cepat, kualitas mantap",pengiriman cepat kualitas mantap
8,"Pengiriman cepat, kualitas mantap",pengiriman cepat kualitas mantap
9,"Standar, tidak ada yang istimewa",standar tidak ada yang istimewa


## 5. Simpan


In [19]:
# TODO: simpan df_encoded ke orders_transformed.csv
df_encoded.to_csv("../orders_transformed.csv", index=False)
